# Testing LLMs for post-OCR error correction

This notebook uses a large language model (LLM) via [Ollama](https://ollama.com) to correct OCR errors extracted in the previous step.

For information on the test case implemented here in addition to its usefulness for research, see the README in `/code`.


## For working in Colab

## Connect to the GPU:
In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'. **NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

## Connect your Google Drive:
Run the code block below to mount your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Set the path to your working directory

In [ ]:
import os

gdrive_path = "/content/drive/My Drive"
wdir_path = os.path.join(gdrive_path, "/THISTLE_project/THISTLE_project")

## Step 1: Install dependencies
Run this cell first to install all required Python packages.

- **In Google Colab:** packages are installed for your current session and will need reinstalling if you reconnect.
- **In Anaconda (local):** run this once per environment, or install via `requirements.txt` instead (see `code/README.md`).



In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pandas ollama tqdm

## 2. Imports and Ollama setup
The following code block imports all dependencies and starts the Ollama server.

In [ ]:
import pandas as pd
import ollama
from tqdm import tqdm
import time
import subprocess
import threading

def run_ollama_serve():
    subprocess.Popen(['ollama', 'serve'])

if 'colab_env' in locals() and colab_env:
    thread = threading.Thread(target=run_ollama_serve)
    thread.start()
    time.sleep(15)
    print('Ollama server started.')

## 3. Load data and concatenate into one dataframe

This code block imports the ground truth data and OCR output extracted in the previous step.

In [ ]:
thistle_df = pd.read_csv('/content/drive/My Drive/THISTLE_project/THISTLE_project/data/ground_truth')
tesseract_df = pd.read_csv('/content/drive/My Drive/THISTLE_project/processed_imgs.csv')


In [ ]:
thistle_df.head()


In [ ]:
tesseract_df.head()

In [ ]:
df = thistle_df.merge(tesseract_df, on="png_img", how="left")

In [ ]:
df.head()

## 4. Define correction function

This code block pulls a model from Ollama, then creates a function that prompts the model to correct the OCR from each row in the dataframe in the 'original_ocr' column and the 'tesseract_ocr' column.

In [ ]:
model_name = 'llama3'
ollama.pull(model_name)

def correct_text(text):
    if pd.isna(text) or text.strip() == '':
        return ''

    # prompt: can be altered
    prompt = (
        "Here is OCR-extracted text from a nineteenth-century newspaper in the public domain. "
        "Please correct OCR errors — such as misread characters, broken words, and garbled punctuation — "
        "so that the text makes grammatical and linguistic sense. "
        "Preserve the historical language and spelling conventions of the original; "
        "do not modernise archaic words or add any text that is not present in the original. "
        f"Return only the corrected text, with no additional commentary.\n\n{text}"
    )

    response = ollama.chat(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1}
    )
    return response['message']['content']

## 5. Run Correction for All Sources

In [ ]:
df['corrected_original'] = [correct_text(t) for t in tqdm(df['original_ocr'])]

df['corrected_tesseract'] = [correct_text(t) for t in tqdm(df['tesseract_text'])]


## 6. Save the results

In [ ]:
df.to_csv('/content/drive/My Drive/THISTLE_project/post_ocr_output.csv', index=False)
